
Medical Chatbot using RAG with LangChain



In [ ]:
# Install Required Libraries
!pip install langchain sentence-transformers

In [ ]:
!pip install chromadb

In [ ]:
!pip install langchain_community pypdf langchain_classic

In [ ]:
!pip install llama-cpp-python

In [ ]:
pip install langchain langchain-core langchain-community

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:

# Load PDF Documents
from langchain_community.document_loaders import PyPDFDirectoryLoader
loader = PyPDFDirectoryLoader("/content/drive/MyDrive/Colab Notebooks")
docs = loader.load()

In [ ]:
# Check Number of Documents
len(docs)

In [ ]:
# Split Documents into Chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
chunks = text_splitter.split_documents(docs)

In [ ]:
# Check Number of Chunks
len(chunks)

In [ ]:
# Set Hugging Face API Token
import os
os.environ["hf_api_token"] = ""

In [ ]:
# Create Embeddings Model
from langchain_community.embeddings import SentenceTransformerEmbeddings
embeddings = SentenceTransformerEmbeddings(model_name="neuml/pubmedbert-base-embeddings")

In [ ]:
# Store Embeddings in ChromaDB
from langchain_community.vectorstores import Chroma
vectorstore = Chroma.from_documents(chunks, embeddings)

In [ ]:
# Perform Similarity Search
query = "what is diabetis"

search_results = vectorstore.similarity_search(query)
search_results

In [ ]:
# Create Retriever
retriever = vectorstore.as_retriever(search_kwargs={"k":5})

In [ ]:
# Retrieve Relevant Chunks
retriever.invoke(query)

In [ ]:
# Load BioMistral LLM Model
from huggingface_hub import hf_hub_download
from langchain_community.llms import LlamaCpp

# 1. Download/get the local path for the GGUF file
model_path = hf_hub_download(
	repo_id="BioMistral/BioMistral-7B-GGUF",
	filename="ggml-model-Q4_K_M.gguf" # Ensure filename matches exactly
)

# 2. Load into the LangChain wrapper
llm = LlamaCpp(
    model_path=model_path,
    temperature=0.7,
    n_ctx=2048,
    # n_gpu_layers=-1, # Use -1 to offload all layers if you have a GPU
)

In [ ]:
# Create Prompt Template
template="""
<|context|>
You are an MedicalAssistant.
Please be truthful and give answers.
</s>
<|user|>
{query}
</s>
<|assistant|>
"""

In [ ]:
# Import Required LangChain Components
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_classic.chains import RetrievalQA, LLMChain

In [ ]:
# Create Prompt
prompt = ChatPromptTemplate.from_template(template)

In [ ]:
# Build RAG Chain
rag_chain = (
    {"context":retriever, "query":RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
    )

In [ ]:
# Generate Response
response = rag_chain.invoke(query)

In [ ]:

# Interactive Chatbot
while True:
    user_input = input("Input query: ")

    if user_input == "exit":
        break

    result = rag_chain.invoke(user_input)

    print("Answer:", result)
